# **4. PLONGEMENT DE DOCUMENTS**

In [ ]:

import re
import umap
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import stopwordsiso as stopwords
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances

from utils import cosine_similarity_matrix, plot_similarity_heatmap, run_umap, analyse_theme_embeddings_moy, analyse_theme_cosinus_moy, select_chunks_for_theme, compute_contributions, plot_umap_projection, compute_pairwise_chunk_similarities, plot_chunk_similarity_heatmap, compute_mean_text_similarities_from_chunk_matrix, plot_text_similarity_heatmap, run_umap_on_text_similarities

In [2]:
df_chunks = pd.read_excel('data/df_chunks_part2_detailed.xlsx')
df_chunks.head()

,AUTEUR,LANGUE,VILLE,CHUNK_MOTS,NB_MOTS,CHUNK_ID,distance avec les autres villes,types de population,esthétique de la ville,histoire de la toponymie urbaine,...,mosquée,église,synagogue,cimetière et tombes,monuments et vestiges,pyramide,jardin,climat,animaux,plantes et arbres
0,AL YAQUBI,arabe,Le Caire,الغسطاط تعرف بباب اليون وهو الموضع المعروف بال...,50,1,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,AL YAQUBI,arabe,Le Caire,عمرو بن العاص مسجد جامعها ودار امارتها المعروف...,36,2,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
2,AL YAQUBI,arabe,Le Caire,واسكنه قوما وكتب الى عمر بن الخطاب بذلك فكتب ا...,50,3,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,AL YAQUBI,arabe,Le Caire,لان لكل كورة مدينة مخصوصة بأمر من الامور من مد...,56,4,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
4,AL YAQUBI,arabe,Le Caire,ومدينة القيس وبها تعل الثياب القيسية والأكسية ...,50,5,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True


In [3]:
# Chargement des themes depuis le fichier texte
with open("data/themes.txt", "r", encoding="utf-8") as f:
    themes = [line.strip() for line in f if line.strip()]

print(themes)

['distance avec les autres villes', 'types de population', 'esthétique de la ville', 'histoire de la toponymie urbaine', 'élement géomorphologique', 'périphérie de ville', 'porte', 'voirie', 'pouvoir', 'palais ; citadelle et château', 'mur et murailles', 'matériaux', 'infrastructure de services', 'eau potable et fontaine', 'pont', 'port et bateau', 'habitation et maison', 'marché', 'richesse de la ville ou du prince', 'impôts et taxes', 'espaces agricoles', 'artisanat', 'mosquée', 'église', 'synagogue', 'cimetière et tombes', 'monuments et vestiges', 'pyramide', 'jardin', 'climat', 'animaux', 'plantes et arbres']


### **4.1. Création des embeddings**

In [8]:
# Nettoyer le corpus
# Dictionnaire langue : ISO
langue_to_iso = {
    'arabe': 'ar',
    'hébreu': 'he',
    'latin': 'la',
    'persan': 'fa'
}

# Application directe
df_chunks["CHUNK_SANS_STOPWORDS"] = df_chunks.apply(
    lambda row: ' '.join([
        word for word in re.findall(r'\w+|\W+', row['CHUNK_MOTS'], re.UNICODE)
        if not (
            word.strip().isalnum() and
            langue_to_iso.get(row['LANGUE']) in stopwords.langs() and
            word.strip() in stopwords.stopwords(langue_to_iso[row['LANGUE']])
        )]),
    axis=1)

In [10]:
df_chunks['NB_MOTS_SANS_STOPWORDS'] = df_chunks['CHUNK_SANS_STOPWORDS'].apply(lambda x: len(x.split()))
df_chunks[['CHUNK_MOTS','NB_MOTS','CHUNK_SANS_STOPWORDS', 'NB_MOTS_SANS_STOPWORDS']].head(10)

,CHUNK_MOTS,NB_MOTS,CHUNK_SANS_STOPWORDS,NB_MOTS_SANS_STOPWORDS
0,الغسطاط تعرف بباب اليون وهو الموضع المعروف بال...,50,الغسطاط تعرف بباب اليون الموضع الم...,37
1,عمرو بن العاص مسجد جامعها ودار امارتها المعروف...,36,عمرو العاص مسجد جامعها ودار امارته...,31
2,واسكنه قوما وكتب الى عمر بن الخطاب بذلك فكتب ا...,50,واسكنه قوما وكتب عمر الخطاب بذلك...,41
3,لان لكل كورة مدينة مخصوصة بأمر من الامور من مد...,56,لان لكل كورة مدينة مخصوصة بأمر ا...,52
4,ومدينة القيس وبها تعل الثياب القيسية والأكسية ...,50,ومدينة القيس وبها تعل الثياب القيسية...,45
5,فى لجانب الشرقى من النيل ومدينة الأشمونين وبها...,53,لجانب الشرقى النيل ومدينة الأشموني...,44
6,ولهما ساحل وبها يعمل الفرش القطوع والجلود الاخ...,28,ولهما ساحل وبها يعمل الفرش القطوع ...,24
7,عل ارضيه فغرقها حتى يختلفوا الى القرى في الزوا...,57,عل ارضيه فغرقها يختلفوا القرى ...,45
8,الى المسجد الجامع بأيديهم الرياحين ويقفون على ...,39,المسجد الجامع بأيديهم الرياحين ويقفو...,30
9,وهناك بساتينهم وضياعهم ومتنزهاتهم وقد عقد على ...,50,وهناك بساتينهم وضياعهم ومتنزهاتهم عق...,35


In [ ]:
# Charger LaBSE
model = SentenceTransformer("sentence-transformers/LaBSE")

def get_labse_embeddings(texts, batch_size=50):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Traitement des batches"):
        batch = texts[i:i+batch_size]
        batch_embeddings = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

X_embeddings = get_labse_embeddings(df_chunks["CHUNK_SANS_STOPWORDS"].tolist(), batch_size=128)
np.save("data/embeddings_LaBSE.npy", X_embeddings)

## **Checkpoint : import de la matrice d'embeddings**

In [4]:
X_embeddings = np.load("data/embeddings_LaBSE.npy") # Import du jeu de données

In [5]:
df_chunks["EMBEDDING"] = list(X_embeddings) #Ajout des embeddings dans le dataframe df_chunks sous forme de liste

## **4.2 Analayse des embeddings des mêmes catégories**

### *4.2.1. Embeddings moyennées*

In [ ]:
# Génération des matrices de similarité et des projections UMAP pour l'ensemble des thèmes
for theme in themes:
    if theme != 'synagogue':
        print(f'Traitement du thème : {theme}')
        analyse_theme_embeddings_moy(df_chunks=df_chunks, theme=theme, base_path='figures/4-2-1_embeddings-moy', show=False)

### *4.2.2. Similarités cosinus moyennées*

In [ ]:
for theme in themes:
    if theme != 'synagogue':
        print(f'Traitement du thème : {theme}')
        analyse_theme_cosinus_moy(df_chunks, theme, base_path='figures/4-2-2_sim-cosin-moy', show=False)

In [12]:
def afficher_chunks(df, auteur, ville, theme):
    # Filtrer selon l'auteur, la ville et la valeur True dans la colonne du thème
    resultats = df[
        (df['AUTEUR'] == auteur) &
        (df['VILLE'] == ville) &
        (df[theme] == True)
    ]

    # Afficher uniquement les colonnes utiles
    return resultats[['CHUNK_MOTS']]

auteur = "AL YAQUBI" #@param['AL YAQUBI', 'IBN  RUSTAH', 'AL-MASUDI', 'IBN HAWQAL', 'AL-MUQADDASI', 'AL-IDRISI', 'HUGUES FALCAND', 'AL-HIMYARI', 'NASIR-I KHOSRWO', 'AL GHARNATI', 'BENJAMIN DE TUDELE', 'IBN JUBAYR', 'THETMAR', 'GERARDUS BURCHARDUS', 'GUILLAUME DE TYR', 'OLIVIER', 'JACQUES DE VITRY', 'VINCENT DE BEAUVAIS', 'IBN SAID']
ville = "Le Caire" #@param["Le Caire","Palerme","Cordoue"]
theme = "élement géomorphologique" # @param ["distance avec les autres villes","types de population","esthétique de la ville","histoire de la toponymie urbaine","élement géomorphologique","périphérie de ville","porte","voirie","pouvoir","palais ; citadelle et château","mur et murailles","matériaux","infrastructure de services","eau potable et fontaine","pont","port et bateau","habitation et maison","marché","richesse de la ville ou du prince","impôts et taxes","espaces agricoles","artisanat","mosquée","église","synagogue","cimetière et tombes","monuments et vestiges","pyramide","jardin","climat","animaux","plantes et arbres"]

chunks = afficher_chunks(df_chunks, auteur, ville, theme)
chunks["CHUNK_MOTS"].tolist()

[]

## **4.3. Analyse des embddings des mêmes textes**

In [25]:
# Étape 1 : Moyenne des embeddings par AUTEUR pour une VILLE donnée
def mean_embedding_by_author_for_city(df, ville):
    filtered = df[df["VILLE"] == ville]
    grouped = filtered.groupby("AUTEUR")["EMBEDDING"]\
                .apply(lambda x: np.mean(np.vstack(x), axis=0))
    return grouped

In [ ]:
# Exemple d'exécution
ville = "Le Caire"

In [ ]:
# Étape 1
mean_embs = mean_embedding_by_author_for_city(df_chunks, ville)

# Étape 2 : Matrice de similarité
df_similarity = cosine_similarity_matrix(mean_embs)
plot_similarity_heatmap(df_similarity, ville, base_path='figures/4-3_embeddings-textes', show=False)

# Étape 3 : UMAP
X_umap_embedding = run_umap(mean_embs)

# Étape 4 : Contributions
contribs, cos2 = compute_contributions(X_umap_embedding)

# Résultat dans un DataFrame
resUMAP_embedding = pd.DataFrame({
    "AUTEUR": mean_embs.index,
    "LANGUE": [df_chunks[df_chunks["AUTEUR"] == auteur]["LANGUE"].iloc[0] for auteur in mean_embs.index],
    "Coord1": X_umap_embedding[:, 0],
    "Contrib1": contribs[:, 0],
    "Cos1": cos2[:, 0],
    "Coord2": X_umap_embedding[:, 1],
    "Contrib2": contribs[:, 1],
    "Cos2": cos2[:, 1],
})

# Affichage UMAP : juste les noms d'auteurs
plot_umap_projection(X_umap_embedding,
                     resUMAP_embedding["AUTEUR"].values,
                     contribs,
                     theme=ville,
                     show=False,
                     base_path='figures/4-3_embeddings-textes',
                     color_labels=resUMAP_embedding["LANGUE"],
                     title=f"Projection UMAP des auteurs pour la ville : {ville}")
